# Rekam medis: LoRA scaling experiments di Google Colab (GPU T4)
Eksperimen skala kecil (model < 100M parameter) tentang kapasitas model, weight decay, pengulangan data, learning rate, dan LoRA. **Tidak** membuktikan scaling law triliunan parameter.

> Colab: buka panel **Daftar isi** di sidebar kiri (ikon ☰) untuk navigasi otomatis dari heading, atau pakai tabel di bawah.

## Daftar isi
| # | Sprint | Isi | Gerbang lulus |
|---|---|---|---|
| 0 | [Persiapan runtime & proyek](#sprint-0) | GPU T4, Drive, instalasi | GPU terdeteksi, proyek ada di `/content` |
| 1 | [Diagnostik & validasi model](#sprint-1) | hardware, dependensi, 4 model diunduh | smoke test semua model `ok: True` |
| 2 | [Dataset & pilot](#sprint-2) | tokenisasi, pilot + uji interrupt/resume | `PASSED` (wajib sebelum sprint 3) |
| 3 | [Eksperimen B + D](#sprint-3) | weight decay, learning rate | 5 run `completed` |
| 4 | [Eksperimen E + A](#sprint-4) | rank LoRA, kapasitas model | run baru `completed` |
| 5 | [Eksperimen C](#sprint-5) | pengulangan data 1x/2x/4x (paling lama) | run rep2 dan rep4 `completed` |
| 6 | [Evaluasi & laporan](#sprint-6) | grafik, tabel, `report.md`, unduh hasil | laporan terbentuk |

**Aturan umum**
- Jalankan sel dari atas ke bawah; selesaikan satu sprint (cek gerbangnya) sebelum lanjut.
- Sesi terputus? Jalankan ulang semua sel Sprint 0, lalu sel sprint yang terhenti: run selesai dilewati, run setengah jadi dilanjutkan dari checkpoint (disimpan di Drive).
- Jangan campur hasil CPU (laptop) dan GPU dalam satu perbandingan; GPU tidak deterministik bit-per-bit.
- Data default sintetis. Jangan unggah rekam medis asli tanpa de-identifikasi nyata (nama dan alamat tidak dihapus otomatis).

## Rencana sprint, langkah demi langkah
**Sprint 0: Persiapan** (sekali per sesi)
1. Runtime → Change runtime type → **T4 GPU**.
2. Letakkan `colab_upload.zip` di `MyDrive/` (atau unggah saat diminta).
3. Jalankan sel GPU, mount Drive + ekstrak proyek, instal paket, tetapkan `SET`.

**Sprint 1: Diagnostik & model**
1. Catat spesifikasi runtime (`hardware_monitor`) dan versi paket.
2. Unduh dan uji `pythia-14m/31m/70m` dan `distilgpt2` (lisensi, tokenizer, estimasi memori, uji forward/backward).
3. Model yang melebihi anggaran otomatis ditolak.

**Sprint 2: Dataset & pilot**
1. Tokenisasi dan ringkasan dataset (split seed 42 per record, tanpa kebocoran).
2. Pilot: latih 8 langkah, hentikan, resume, selesaikan. Lalu `--estimate` untuk perkiraan waktu dari kecepatan terukur.
3. Pakai string `SET` yang sama di semua sprint.

**Sprint 3: B + D** (murah, 5 run; cek apakah ada sinyal)
1. Dry-run untuk melihat rencana run.
2. Jalankan; amati val loss vs weight decay dan vs learning rate.

**Sprint 4: E + A**
1. E: rank LoRA 4/8/16 (laporan persentase parameter trainable).
2. A: pythia-14m/31m/70m vs distilgpt2 (bandingkan dengan bits/char, bukan loss mentah, karena tokenizer berbeda).

**Sprint 5: C** (paling berat)
1. Pengulangan data 2x dan 4x pada 1 epoch (langkah optimizer ikut bertambah).
2. Periksa apakah train loss turun sementara val loss tidak (tanda overfitting).

**Sprint 6: Evaluasi**
1. Bangun grafik dan `report.md`, tampilkan di notebook.
2. Unduh `hasil_colab.zip` (laporan, grafik, CSV, metadata).

<a name="sprint-0"></a>
## Sprint 0: Persiapan runtime & proyek
**Runtime → Change runtime type → T4 GPU**, lalu jalankan sel berikut.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU tidak aktif: Runtime > Change runtime type > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
!nvidia-smi

Ambil proyek dan simpan keluaran ke Drive. Letakkan `colab_upload.zip` di `MyDrive/`, atau unggah manual saat diminta.

In [ ]:
import os, zipfile, pathlib
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/rekam-medis-ml')   # semua keluaran persisten di sini
PROJ = pathlib.Path('/content/rekam-medis-machine-learning')
DRIVE.mkdir(parents=True, exist_ok=True)
zip_path = pathlib.Path('/content/drive/MyDrive/colab_upload.zip')
if not zip_path.exists():
    from google.colab import files
    up = files.upload(); zip_path = pathlib.Path('/content') / next(iter(up))
if not PROJ.exists():
    zipfile.ZipFile(zip_path).extractall(PROJ)
os.chdir(PROJ)
# keluaran -> Drive (symlink) agar resume bekerja setelah disconnect
for d in ['experiments', 'checkpoints', 'models/adapters', 'reports', 'data/processed']:
    target = DRIVE / d; target.mkdir(parents=True, exist_ok=True)
    p = PROJ / d
    if p.exists() and not p.is_symlink():
        import shutil; shutil.rmtree(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists(): p.symlink_to(target)
print(os.getcwd(), os.listdir('.'))

In [ ]:
!pip -q install -U peft wandb psutil pyyaml pandas matplotlib huggingface_hub

In [ ]:
# Pengaturan GPU yang dipakai SEMUA perintah di bawah (harus konsisten agar resume valid).
# Sesi baru: jalankan sel ini lagi sebelum sprint mana pun.
SET = ("hardware.device=cuda hardware.num_threads=2 hardware.max_train_ram_gb=10 hardware.min_available_ram_gb=1.5 "
       "train.batch_size=4 train.grad_accum=1 train.eval_batch_size=16")
print(SET)

<a name="sprint-1"></a>
## Sprint 1: Diagnostik & validasi model
Uji forward/backward di sel model-check berjalan di CPU; kecocokan GPU diuji oleh pilot di Sprint 2.

In [ ]:
!python -m src.hardware_monitor
!python -m src.model_loader --check-deps

In [ ]:
!python -m src.model_loader --models EleutherAI/pythia-14m EleutherAI/pythia-31m EleutherAI/pythia-70m distilgpt2 --download --smoke-test

<a name="sprint-2"></a>
## Sprint 2: Dataset & pilot (gerbang wajib)

In [ ]:
!python -m src.dataset_processor

In [ ]:
!python -m src.scaling_experiments --pilot --set {SET}

In [ ]:
!python -m src.scaling_experiments --estimate --set {SET}

<a name="sprint-3"></a>
## Sprint 3: Eksperimen B (weight decay) + D (learning rate)
W&B opsional: jalankan `wandb.login()` sekali untuk mode online; tanpa itu mode `auto` memakai offline.

In [ ]:
# import wandb; wandb.login()   # opsional; masukkan API key secara interaktif, jangan simpan di notebook

In [ ]:
!python -m src.scaling_experiments --experiments B D --dry-run --set {SET}

In [ ]:
!python -m src.scaling_experiments --experiments B D --set {SET}

<a name="sprint-4"></a>
## Sprint 4: Eksperimen E (rank LoRA) + A (kapasitas model)

In [ ]:
!python -m src.scaling_experiments --experiments E A --dry-run --set {SET}

In [ ]:
!python -m src.scaling_experiments --experiments E A --set {SET}

<a name="sprint-5"></a>
## Sprint 5: Eksperimen C (pengulangan data 1x / 2x / 4x)
Paling lama: run 4x melatih 4 kali lebih banyak langkah.

In [ ]:
!python -m src.scaling_experiments --experiments C --dry-run --set {SET}

In [ ]:
!python -m src.scaling_experiments --experiments C --set {SET}

<a name="sprint-6"></a>
## Sprint 6: Evaluasi & laporan

In [ ]:
!python -m src.evaluation

In [ ]:
from IPython.display import Markdown
Markdown(open('reports/report.md', encoding='utf-8').read())

In [ ]:
!cd reports && zip -qr /content/hasil_colab.zip . && echo ok
from google.colab import files; files.download('/content/hasil_colab.zip')